# NB_lab_test_trends - Silver to Gold

Builds `<catalog>.gold.lab_test_trends` (one row per test_name per calendar month) per the STTM `sttm_silver_to_gold.xlsx`. Self-contained aggregation of `silver.lab_results`, optional LEFT JOIN to the calendar dimension for month/quarter labels, and a LAG() month-over-month abnormal-rate trend. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_hospital.gold;

CREATE OR REPLACE TABLE adb_hospital.gold.lab_test_trends AS

WITH monthly AS (
    SELECT
        test_name,
        YEAR(test_date) AS year,
        MONTH(test_date) AS month,
        COUNT(*) AS total_tests,
        COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical') AS critical_test_count
    FROM adb_hospital.silver.lab_results
    GROUP BY
        test_name,
        YEAR(test_date),
        MONTH(test_date)
),

rates AS (
    SELECT
        *,
        ROUND(abnormal_test_count * 100.0 / total_tests, 2) AS abnormal_rate_pct,
        ROUND(critical_test_count * 100.0 / total_tests, 2) AS critical_rate_pct
    FROM monthly
),

trend AS (
    SELECT
        *,
        LAG(abnormal_rate_pct) OVER (
            PARTITION BY test_name
            ORDER BY year, month
        ) AS prev_month_abnormal_rate_pct
    FROM rates
)

SELECT
    t.test_name,
    t.year,
    t.month,
    c.month_name,
    c.quarter_name,
    t.total_tests,
    t.abnormal_test_count,
    t.critical_test_count,
    t.abnormal_rate_pct,
    t.critical_rate_pct,
    t.prev_month_abnormal_rate_pct,
    ROUND(
        t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct,
        2
    ) AS abnormal_rate_change_pct

FROM trend t

LEFT JOIN adb_hospital.silver.calender c
    ON c.date_key = CAST(
        DATE_FORMAT(
            MAKE_DATE(t.year, t.month, 1),
            'yyyyMMdd'
        ) AS INT
    );

In [0]:
total_count = spark.table("adb_hospital.gold.lab_test_trends").count()
dbutils.notebook.exit(str(total_count))